# 🔬 OpenAmadeus Companion — Kurisu Qwen3-TTS 1.7B Remote Worker (Acelerado)

Este notebook executa o modelo **Kurisu Qwen3-TTS 1.7B** (`starrydark/Kurisu_Qwen3_TTS`) acelerado com **CUDA Graphs** (`faster-qwen3-tts`) em GPUs em nuvem (**Google Colab Pro L4 / A100 / T4**), gerando um túnel HTTPS seguro via **Cloudflare Tunnel** com latência em tempo real (< 400ms TTFA).

### Melhorias Técnicas de Tempo Real:
- **Aceleração CUDA Graphs:** Reduz o passo de inferência do talker de ~330ms para ~54ms (RTF > 1.9x).
- **Streaming Contínuo (`audio/L16;rate=24000`):** Primeiro chunk de áudio emitido em ~390ms.
- **Suavização Contínua $C^0$ nas Costuras:** Rampa de 120 amostras nas bordas dos chunks para eliminar micro-estalos sem perda de amplitude.
- **Warm-up Completo:** Pré-aquecimento do Transformer e do decodificador Mimi, garantindo resposta imediata na 1ª fala.

In [ ]:
# Passo 1: Verificar GPU disponível no Colab
!nvidia-smi

In [ ]:
# Passo 2: Instalar dependências fixadas para CUDA Graphs e Qwen3-TTS
!apt-get update -qq && apt-get install -qq -y sox libsox-fmt-all
!pip install -q --upgrade pip
!pip install -q "transformers==5.15.1" faster-qwen3-tts soundfile fastapi uvicorn pydantic pycloudflared httpx

In [ ]:
# Passo 3: Iniciar Servidor Kurisu Qwen3-TTS com CUDA Graphs e Streaming Estabilizado
import os
import io
import re
import time
import torch
import soundfile as sf
import numpy as np
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, StreamingResponse
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare
from faster_qwen3_tts import FasterQwen3TTS

print("=" * 65)
print("⏳ Carregando Kurisu Qwen3-TTS 1.7B com faster-qwen3-tts...")
print("=" * 65)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

t_load_start = time.perf_counter()
model = FasterQwen3TTS.from_pretrained(
    "starrydark/Kurisu_Qwen3_TTS",
    device=device,
    dtype=dtype,
    max_seq_len=2048,
)
print(f"✅ Modelo carregado na GPU em {round(time.perf_counter() - t_load_start, 2)}s")

# Calibrar PredictorGraph (codebooks acústicos 1..15) ANTES da captura do CUDA Graph
if hasattr(model, "predictor_graph") and model.predictor_graph is not None:
    model.predictor_graph.temperature = 0.6
    model.predictor_graph.top_p = 0.9
    print("✅ PredictorGraph calibrado (temperature=0.6, top_p=0.9)")

# Aquecer Talker e compilar CUDA Graphs
print("⚡ Aquecendo modelo e compilando CUDA Graphs...")
t_warmup = time.perf_counter()
model.warmup(prefill_len=100)
print(f"✅ Warmup de CUDA Graphs concluído em {round(time.perf_counter() - t_warmup, 2)}s")

# Warm-up completo incluindo o decodificador Mimi
print("🔥 Aquecendo decodificador de áudio Mimi com chunk de teste...")
warmup_gen = model.generate_custom_voice_streaming(
    text="Oi.",
    speaker="kurisu",
    language="Portuguese",
    instruct=None,
    chunk_size=8,
)
for _ in warmup_gen:
    break
print("✅ Pipeline 100% aquecido! TTFA pronto em ~390ms.")

def apply_boundary_continuity(chunk: np.ndarray, prev_sample: Optional[float], ramp_len: int = 120) -> np.ndarray:
    if prev_sample is None or len(chunk) == 0:
        return chunk
    actual_ramp = min(ramp_len, len(chunk))
    start_jump = float(prev_sample) - float(chunk[0])
    ramp = np.linspace(1.0, 0.0, actual_ramp, dtype=chunk.dtype)
    chunk[:actual_ramp] += start_jump * ramp
    return chunk

def sanitize_for_qwen(text: str) -> str:
    s = re.sub(r"<!--[^>]*-->", "", text or "")
    s = re.sub(r"\*[^*]+\*", "", s)
    s = re.sub(r"[*_~`#\"'«»]", "", s)
    s = re.sub(r"[…‥]", "...", s)
    s = re.sub(r"\?+!+[!?]*|!+\?+[!?]*", "?", s)
    s = re.sub(r"!+", ".", s)
    s = re.sub(r"([.,])\s*\.", r"\1", s)
    s = re.sub(
        r"(?i)\b(?:[u]+(?:\s*\.{2,}\s*|-|\s+)[u]+m|[u]+h+m*|[u]+(?:\s*\.{2,}\s*|-|\s+)uh|h+u*m+)\b\s*[,;:.]*\s*",
        "",
        s,
    )
    def _un_stutter(m: re.Match) -> str:
        prefix, word = m.group(1), m.group(2)
        if prefix.lower() == word[0].lower():
            return word.capitalize() if prefix.isupper() else word
        return m.group(0)
    s = re.sub(r"\b([A-Za-zÀ-ÿ])(?:\s*\.{2,}\s*|-)([a-zà-ÿ]+)\b", _un_stutter, s)
    s = re.sub(r"\s*\.{2,}\s*$", ".", s)
    s = re.sub(r"\s*\.{2,}\s*", ", ", s)
    s = re.sub(r",\s*,+", ",", s)
    s = re.sub(r"([.?!])\s*,", r"\1", s)
    s = re.sub(r",\s*([.?!])", r"\1", s)
    s = re.sub(r"^[,;:\s]+", "", s)
    s = re.sub(r"\s+([.,?!])", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s and s[0].islower():
        s = s[0].upper() + s[1:]
    if s and s[-1] not in ".?":
        s += "."
    return s

def prepare_worker_segments(text: str, max_single_pass_chars: int = 130) -> list[str]:
    cleaned = sanitize_for_qwen(text)
    if not cleaned:
        return []
    if len(cleaned) <= max_single_pass_chars:
        return [cleaned]
    raw_parts = [p.strip() for p in re.split(r"(?<=[.?!])\s+", cleaned) if p.strip()]
    segments = []
    for part in raw_parts:
        if part[-1] not in ".?":
            part += "."
        if segments and (len(segments[-1]) + 1 + len(part) <= max_single_pass_chars or len(part) < 25):
            segments[-1] = f"{segments[-1]} {part}"
        else:
            segments.append(part)
    return segments or [cleaned]

app = FastAPI(title="OpenAmadeus Qwen3-TTS Colab Worker")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SynthesizeRequest(BaseModel):
    text: str
    language: Optional[str] = "Portuguese"
    instruct: Optional[str] = None
    temperature: Optional[float] = 0.65
    top_p: Optional[float] = 0.90
    repetition_penalty: Optional[float] = 1.05
    max_new_tokens: Optional[int] = 2048

@app.get("/")
def root():
    return {"system": "OpenAmadeus Qwen3-TTS Colab Worker", "status": "ready", "backend": "faster_qwen3_tts"}

@app.get("/health")
def health():
    vram = torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
    return {
        "status": "ready",
        "device": device,
        "backend": "faster_qwen3_tts",
        "cuda_available": torch.cuda.is_available(),
        "vram_allocated_mb": round(vram, 2),
        "model": "starrydark/Kurisu_Qwen3_TTS",
    }

@app.post("/synthesize_stream")
def synthesize_stream(req: SynthesizeRequest):
    segments = prepare_worker_segments(req.text or "")
    if not segments:
        raise HTTPException(status_code=400, detail="Texto vazio.")

    eff_temp = min(req.temperature if req.temperature is not None else 0.65, 0.65)
    eff_top_p = min(req.top_p if req.top_p is not None else 0.90, 0.92)
    eff_rep = max(req.repetition_penalty if req.repetition_penalty is not None else 1.05, 1.05)
    eff_max_tokens = req.max_new_tokens if req.max_new_tokens is not None else 2048

    def pcm_generator():
        prev_sample = None
        with torch.inference_mode():
            for seg_idx, seg_text in enumerate(segments):
                if seg_idx > 0:
                    pause_chunk = np.zeros(int(24000 * 0.18), dtype=np.float32)
                    pause_chunk = apply_boundary_continuity(pause_chunk, prev_sample)
                    prev_sample = float(pause_chunk[-1])
                    yield (np.clip(pause_chunk, -1.0, 1.0) * 32767).astype(np.int16).tobytes()

                stream_gen = model.generate_custom_voice_streaming(
                    text=seg_text,
                    speaker="kurisu",
                    language=req.language or "Portuguese",
                    instruct=None,
                    chunk_size=8,
                    temperature=eff_temp,
                    top_p=eff_top_p,
                    repetition_penalty=eff_rep,
                    max_new_tokens=eff_max_tokens,
                )
                for audio_chunk, sr, _ in stream_gen:
                    chunk = audio_chunk.copy()
                    chunk = apply_boundary_continuity(chunk, prev_sample)
                    prev_sample = float(chunk[-1])
                    pcm16 = (np.clip(chunk, -1.0, 1.0) * 32767).astype(np.int16).tobytes()
                    yield pcm16

    return StreamingResponse(
        pcm_generator(),
        media_type="audio/L16;rate=24000",
        headers={
            "X-Audio-Format": "pcm_s16le",
            "X-Sample-Rate": "24000",
            "X-Backend": "faster_qwen3_tts",
            "X-Accel-Buffering": "no",
        },
    )

@app.post("/synthesize")
def synthesize(req: SynthesizeRequest):
    segments = prepare_worker_segments(req.text or "")
    if not segments:
        raise HTTPException(status_code=400, detail="Texto vazio.")

    eff_temp = min(req.temperature if req.temperature is not None else 0.65, 0.65)
    eff_top_p = min(req.top_p if req.top_p is not None else 0.90, 0.92)
    eff_rep = max(req.repetition_penalty if req.repetition_penalty is not None else 1.05, 1.05)
    eff_max_tokens = req.max_new_tokens if req.max_new_tokens is not None else 2048

    t0 = time.perf_counter()
    assembled_pieces = []
    prev_sample = None
    sr = 24000

    with torch.inference_mode():
        for seg_idx, seg_text in enumerate(segments):
            if seg_idx > 0:
                pause_chunk = np.zeros(int(sr * 0.18), dtype=np.float32)
                pause_chunk = apply_boundary_continuity(pause_chunk, prev_sample)
                prev_sample = float(pause_chunk[-1])
                assembled_pieces.append(pause_chunk)

            wavs, sr = model.generate_custom_voice(
                text=seg_text,
                speaker="kurisu",
                language=req.language or "Portuguese",
                instruct=None,
                temperature=eff_temp,
                top_p=eff_top_p,
                repetition_penalty=eff_rep,
                max_new_tokens=eff_max_tokens,
            )
            seg_audio = wavs[0]
            if isinstance(seg_audio, torch.Tensor):
                seg_audio = seg_audio.detach().cpu().numpy()
            if seg_audio.ndim > 1:
                seg_audio = seg_audio.squeeze()
            seg_audio = seg_audio.astype(np.float32, copy=True)
            seg_audio = apply_boundary_continuity(seg_audio, prev_sample)
            if len(seg_audio) > 0:
                prev_sample = float(seg_audio[-1])
            assembled_pieces.append(seg_audio)

    audio = np.concatenate(assembled_pieces) if len(assembled_pieces) > 1 else assembled_pieces[0]
    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    buf.seek(0)
    elapsed = round(time.perf_counter() - t0, 3)
    print(f"🎙️ Síntese ({len(segments)} segs): {len(req.text)} caracteres em {elapsed}s")
    return Response(
        content=buf.read(),
        media_type="audio/wav",
        headers={"X-Backend": "faster_qwen3_tts"},
    )

# Abrir Túnel Cloudflare na porta 8005
PORT = 8005
tunnel_res = try_cloudflare(port=PORT)
tunnel_url = tunnel_res.tunnel if hasattr(tunnel_res, 'tunnel') else str(tunnel_res)
print("\n" + "=" * 65)
print("  🌟 URL PÚBLICA DO CLOUDFLARE TUNNEL (CUDA Graphs Real-Time):")
print(f"  👉 {tunnel_url}")
print("=" * 65)
print("  Copie e cole este link em 'Configurações > Voz > URL Remota (Colab)' no OpenAmadeus Companion!\n")

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)
await server.serve()